# 02 · Chất lượng tín hiệu và quyết định preprocessing
**Câu hỏi:** ta loại ECG nào, giữ ECG nào dù có chú thích nhiễu, và một phép lọc thay đổi tín hiệu ra sao?

**Đầu vào:** nguồn 500 Hz + metadata. **Đầu ra:** `.npy` vật lý mV và `records.csv` ghi mọi ECG. **Tiếp theo:** 03 fit tham số chuẩn hóa từ train.

Baseline: **minimal, 500 Hz, 12 lead × 5.000 điểm, float32**. Không mặc định bandpass, cắt biên độ, nội suy điểm hỏng hoặc z-score riêng từng ECG. Minimal là mốc so sánh giữ thông tin, không có nghĩa toàn bộ dữ liệu sạch nhiễu.

**Bản sửa 01/11/2026 chưa được chạy toàn bộ.** Các bảng/hình được tính khi Line chạy. Mục 1 xử lý toàn bộ nguồn; phần phân tích sau đó chỉ đọc train và giới hạn số waveform.

In [ ]:
from pathlib import Path
import os
import sys

def tim_thu_muc_project():
    explicit = os.environ.get("PTBXL_PROJECT_ROOT")
    candidates = [Path(explicit).expanduser().resolve()] if explicit else [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "ptbxl_pipeline").is_dir() and (candidate / "requirements-ptbxl.txt").is_file():
            return candidate.resolve()
    raise FileNotFoundError("Không tìm thấy project. Đặt PTBXL_PROJECT_ROOT tới thư mục chứa ptbxl_pipeline và requirements-ptbxl.txt.")

ROOT = tim_thu_muc_project()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RAW_DIR = Path(os.environ.get("PTBXL_RAW_DIR", str(ROOT / "data/raw/ptbxl/1.0.3"))).expanduser().resolve()
CACHE_DIR = ROOT / "data/cache"
OUTPUT_DIR = Path(os.environ.get("PTBXL_OUTPUT_DIR", str(ROOT / "data/processed/ptbxl_v1.0.3_500hz_minimal"))).expanduser().resolve()
REPORT_DIR = ROOT / "artifacts/ptbxl_reports" / OUTPUT_DIR.name
TARGET_FS = 500
PREPROCESSING = "minimal"
EXPECTED_RECORDS = 21799
print("Project:", ROOT)
print("Dữ liệu gốc:", RAW_DIR)
print("Bộ xuất:", OUTPUT_DIR)

In [ ]:
from ptbxl_pipeline.core import PipelineConfig

config = PipelineConfig(
    raw_dir=RAW_DIR,
    output_dir=OUTPUT_DIR,
    target_fs=TARGET_FS,
    preprocessing=PREPROCESSING,
    expected_records=EXPECTED_RECORDS,
)

## 1. TOÀN BỘ: QC kỹ thuật và xuất tín hiệu
Core đối chiếu nguồn, lead/đơn vị/shape; đổi mV, sắp 12 lead; loại NaN/Inf hoặc lead hằng; ghi từng file và lý do. Không sửa lỗi bằng nội suy/clipping mà không có thí nghiệm.

| Trạng thái | Nghĩa | Supervised training? |
|---|---|---|
| accepted | Có nhãn và qua hợp đồng kỹ thuật | Có |
| no_label | Không có diagnostic superclass | Không; không đổi thành NORM |
| rejected | Lỗi kỹ thuật được ghi lý do | Không; giữ dòng truy vết |

Bước này đọc/ghi nhiều GB. Lỗi I/O/checksum phải được xử lý ở nguồn, không biến thành một loại nhiễu để tiếp tục. Run All sẽ xử lý toàn bộ.

In [ ]:
from IPython.display import display
from ptbxl_pipeline.core import LEADS, CLASSES, process_signals
import pandas as pd

records = process_signals(config)
assert len(records) == EXPECTED_RECORDS and records.ecg_id.is_unique
assert records.status.isin(["accepted", "rejected", "no_label"]).all()
label_columns = [f"y_{name}" for name in CLASSES]
train_records = records.loc[records.split.eq("train")].sort_values("ecg_id").copy()
accepted_train = train_records.loc[train_records.status.eq("accepted")].copy()
assert len(accepted_train) > 0
print("Trạng thái mọi ECG:", OUTPUT_DIR / "records.csv")
display(records.groupby(["split", "status"]).size().unstack(fill_value=0).reindex(["train", "validation", "test"]))

## 2. Tách lỗi nhãn khỏi lỗi tín hiệu
Không gộp mọi ca không được giữ rồi lấy head(10): ca lỗi tín hiệu có thể bị che bởi nhiều ca thiếu nhãn. Bảng toàn bộ lý do là QA; ví dụ chi tiết chỉ xem train.

ECG 12722 được phát hiện trong release 24/10/2026 là ví dụ cần đối chiếu, không phải điều kiện ép dữ liệu phải có đúng một lỗi.

In [ ]:
display(records.groupby(["status", "reason"], dropna=False).size().rename("so_ECG").reset_index())
rejected_train = train_records.loc[train_records.status.eq("rejected")]
no_label_train = train_records.loc[train_records.status.eq("no_label")]
print("Lỗi tín hiệu train:", len(rejected_train))
display(rejected_train[["ecg_id", "reason", "filename_hr", *label_columns]])
print("Thiếu nhãn train:", len(no_label_train), "— chỉ hiển thị 10 ví dụ")
display(no_label_train[["ecg_id", "scp_codes_json", "reason"]].head(10))

## 3. Quan sát ca bị loại
Hàm dưới đây dùng header để sắp lead/đổi mV, không loại lead hằng: cần giữ nguyên lỗi để quan sát. Nó giải thích dữ liệu, không thay core. Nếu có nhiều lỗi train, minh họa một ca xác định; bảng trên vẫn ghi mọi ca.

In [ ]:
import numpy as np

def physical_mV(record, leads):
    """Minh họa đọc header: trả [lead,time] mV, chưa áp dụng QC loại lead hằng."""
    names = [str(name).strip().upper() for name in record.sig_name]
    wanted = [str(name).upper() for name in leads]
    if len(names) != len(set(names)) or not set(wanted).issubset(names):
        raise ValueError("missing/duplicate lead trong header")
    order = [names.index(name) for name in wanted]
    factors = {"mv": 1.0, "uv": 0.001, "µv": 0.001, "μv": 0.001, "v": 1000.0}
    units = [str(record.units[i]).strip().lower() for i in order]
    if any(unit not in factors for unit in units):
        raise ValueError("unsupported unit trong header")
    values = np.asarray(record.p_signal, dtype=np.float64)
    if values.ndim != 2 or values.shape[1] != len(names):
        raise ValueError("shape không khớp header")
    return (values[:, order] * np.asarray([factors[unit] for unit in units])).T

In [ ]:
import wfdb
import matplotlib.pyplot as plt

if rejected_train.empty:
    print("Không có ECG train bị loại kỹ thuật trong lần chạy này.")
else:
    preferred = rejected_train.loc[rejected_train.ecg_id.eq(12722)]
    rejected_example = preferred.iloc[0] if len(preferred) else rejected_train.iloc[0]
    raw_rejected = wfdb.rdrecord(str(RAW_DIR / rejected_example.filename_hr), physical=True)
    rejected_mV = physical_mV(raw_rejected, LEADS)
    spans = np.ptp(rejected_mV, axis=1)
    display(pd.DataFrame({"lead": LEADS, "ptp_mV": spans, "lead_hang": spans == 0}))
    fig, axes = plt.subplots(6, 2, figsize=(15, 10), sharex=True)
    for index, axis in enumerate(axes.flat):
        axis.plot(np.arange(rejected_mV.shape[1]) / raw_rejected.fs, rejected_mV[index], linewidth=0.65)
        axis.set_title(f"{LEADS[index]} — ptp={spans[index]:.4g} mV")
        axis.set_ylabel("mV")
    for axis in axes[-1]:
        axis.set_xlabel("Giây")
    fig.suptitle(f"ECG train {int(rejected_example.ecg_id)} — {rejected_example.reason}")
    fig.tight_layout()
    plt.show()

**Diễn giải:** lead hằng đủ 10 giây là lỗi theo hợp đồng hiện tại. Không dùng biên độ lớn/nhỏ đơn thuần để kết luận lỗi; không thay lead hỏng bằng 0 rồi coi là đủ 12 lead. Chính sách loại có thể gây lệch chọn mẫu; 04 sẽ báo ảnh hưởng theo nhãn.

## 4. Chú thích nhiễu: mẫu số và ý nghĩa
baseline_drift, static_noise, burst_noise, electrodes_problems là chú thích nguồn. Có chú thích không tự động phải loại; trống không chứng minh sạch. Chúng không phải SNR hoặc nhãn chất lượng lâm sàng đã đo lại.

In [ ]:
import pandas as pd

def annotation_present(series):
    """Chuỗi có nội dung; thiếu dữ liệu không được tính là chú thích nhiễu."""
    text = series.astype("string").str.strip().str.lower()
    return (series.notna() & ~text.isin(["", "nan", "none"])).fillna(False)

In [ ]:
noise_columns = ["baseline_drift", "static_noise", "burst_noise", "electrodes_problems"]
assert set(noise_columns).issubset(train_records)
noise_flags = train_records[noise_columns].apply(annotation_present)
noise_rows = []
for cohort_name, mask in (("train trước QC", pd.Series(True, index=train_records.index)), ("train accepted", train_records.status.eq("accepted"))):
    denominator = int(mask.sum())
    flags = noise_flags.loc[mask]
    for name in [*noise_columns, "it_nhat_mot_chu_thich"]:
        count = int(flags.any(axis=1).sum()) if name == "it_nhat_mot_chu_thich" else int(flags[name].sum())
        noise_rows.append({"cohort": cohort_name, "chu_thich": name, "so_ECG": count, "mau_so": denominator, "ty_le_pct": 100 * count / denominator if denominator else np.nan})
display(pd.DataFrame(noise_rows))
print("Các cột chồng nhau; hàng 'ít nhất một' đếm ECG duy nhất.")

## 5. Ví dụ có chủ đích: tránh chỉ nhìn một ECG
Lấy một ECG accepted/train đầu tiên cho mỗi loại chú thích nhiễu và mỗi nhãn có sẵn. Danh sách này không phải mẫu ngẫu nhiên để ước lượng prevalence. Hình chỉ vẽ II/V5 cho tối đa bốn nhóm nhiễu; nhiễu có thể nằm ở lead khác nên phải đọc mô tả nguồn.

In [ ]:
example_rows = []
for name in noise_columns:
    candidates = accepted_train.loc[annotation_present(accepted_train[name])]
    if len(candidates):
        row = candidates.iloc[0]
        example_rows.append({"nhom": name, "ecg_id": int(row.ecg_id), "mo_ta_nguon": row[name]})
for name in CLASSES:
    candidates = accepted_train.loc[accepted_train[f"y_{name}"].eq(1)]
    if len(candidates):
        example_rows.append({"nhom": f"nhãn {name}", "ecg_id": int(candidates.iloc[0].ecg_id), "mo_ta_nguon": "không suy diễn hình thái từ nhãn nhóm"})
display(pd.DataFrame(example_rows))
noise_examples = [item for item in example_rows if item["nhom"] in noise_columns]
if noise_examples:
    fig, axes = plt.subplots(len(noise_examples), 2, figsize=(14, 2.5 * len(noise_examples)), squeeze=False)
    for row_index, item in enumerate(noise_examples):
        row = accepted_train.loc[accepted_train.ecg_id.eq(item["ecg_id"])].iloc[0]
        x = np.load(OUTPUT_DIR / row.npy_path, allow_pickle=False)
        for column_index, lead in enumerate(("II", "V5")):
            axis = axes[row_index, column_index]
            axis.plot(np.arange(x.shape[1]) / TARGET_FS, x[LEADS.index(lead)], linewidth=0.6)
            axis.set(title=f"{item['nhom']} · ECG {item['ecg_id']} · {lead}", xlabel="Giây", ylabel="mV")
    fig.tight_layout()
    plt.show()

## 6. Raw và file xuất có khớp không?
Chọn ECG train có chú thích baseline drift nếu có, nếu không dùng accepted/train đầu tiên. Với 500 Hz/minimal, file xuất phải bằng dữ liệu vật lý đã sắp lead và đổi float32. Với release lọc/giảm mẫu, không mong bằng nhau; cần giải thích phép biến đổi riêng.

In [ ]:
drift_candidates = accepted_train.loc[annotation_present(accepted_train.baseline_drift)]
example = drift_candidates.iloc[0] if len(drift_candidates) else accepted_train.iloc[0]
raw_record = wfdb.rdrecord(str(RAW_DIR / example.filename_hr), physical=True)
raw_mV = physical_mV(raw_record, LEADS)
export_mV = np.load(OUTPUT_DIR / example.npy_path, allow_pickle=False)
assert export_mV.shape == (12, TARGET_FS * 10) and export_mV.dtype == np.float32
assert np.isfinite(export_mV).all()
if TARGET_FS == 500 and PREPROCESSING == "minimal":
    np.testing.assert_array_equal(export_mV, raw_mV.astype(np.float32))
    print("Raw sau đổi float32 khớp export của ECG", int(example.ecg_id))
else:
    print("Release có biến đổi; đồ thị không phải kiểm tra bằng nhau.")
fig, axes = plt.subplots(6, 2, figsize=(15, 12), sharex=True)
for index, axis in enumerate(axes.flat):
    axis.plot(np.arange(raw_mV.shape[1]) / raw_record.fs, raw_mV[index], label="raw mV", linewidth=0.8)
    axis.plot(np.arange(export_mV.shape[1]) / TARGET_FS, export_mV[index], label="export", linewidth=0.6, alpha=0.7)
    axis.set(title=LEADS[index], ylabel="mV")
axes[0, 0].legend()
for axis in axes[-1]:
    axis.set_xlabel("Giây")
fig.suptitle(f"Raw/export — ECG train {int(example.ecg_id)}")
fig.tight_layout()
plt.show()

## 7. Phân bố biên độ trên mẫu train có giới hạn
Đọc tối đa 128 ECG accepted/train, seed cố định. Mỗi lần đọc một waveform; bảng này không chứng nhận toàn bộ train. Mean/std/RMS/peak-to-peak mô tả tín hiệu, không trực tiếp đo chất lượng/bệnh. Cực trị gợi ý kiểm tra thêm, không tự tạo ngưỡng loại.

In [ ]:
import numpy as np
import pandas as pd

def summarize_signal(values, leads):
    values = np.asarray(values, dtype=np.float64)
    if values.ndim != 2 or values.shape[0] != len(leads) or not np.isfinite(values).all():
        raise ValueError("shape/nonfinite trong mẫu phân tích")
    return pd.DataFrame({"lead": leads, "mean_mV": values.mean(axis=1), "std_mV": values.std(axis=1),
                         "ptp_mV": np.ptp(values, axis=1), "rms_mV": np.sqrt(np.mean(values ** 2, axis=1))})

In [ ]:
sample_rows = accepted_train.sample(n=min(128, len(accepted_train)), random_state=42).sort_values("ecg_id")
summary_parts = []
for row in sample_rows.itertuples(index=False):
    table = summarize_signal(np.load(OUTPUT_DIR / row.npy_path, allow_pickle=False), LEADS)
    table["ecg_id"] = int(row.ecg_id)
    summary_parts.append(table)
signal_summary = pd.concat(summary_parts, ignore_index=True)
print(f"Phạm vi: {len(sample_rows)}/{len(accepted_train)} ECG accepted/train, seed=42; chưa phân tích biên độ mọi waveform.")
display(signal_summary.groupby("lead", sort=False)[["std_mV", "ptp_mV", "rms_mV"]].agg(["median", "max"]).reindex(LEADS))
display(signal_summary.nlargest(10, "ptp_mV")[["ecg_id", "lead", "ptp_mV", "rms_mV"]])
fig, axis = plt.subplots(figsize=(12, 4))
for index, lead in enumerate(LEADS):
    values = signal_summary.loc[signal_summary.lead.eq(lead), "ptp_mV"]
    axis.scatter(np.full(len(values), index), values, s=8, alpha=0.35)
axis.set_xticks(range(len(LEADS)), LEADS)
axis.set(ylabel="Peak-to-peak (mV)", title=f"Biên độ từng lead trên {len(sample_rows)} ECG train lấy mẫu")
plt.show()

## 8. Trong RAM: minimal, bandpass và 200 Hz
Giữ cùng ECG raw 500 Hz. Tạo ba góc nhìn: nguyên bản; Butterworth bậc thiết kế 4, SOS hai chiều 0,5–40 Hz; giảm 500→200 Hz bằng polyphase FIR có chống aliasing. Đây là hai phép biến đổi riêng, không ngầm kết hợp lọc 40 Hz với giảm mẫu.

Hai chiều tránh độ trễ pha tổng thể nhưng có ảnh hưởng biên và thay đổi đáp ứng biên độ. 0,5 Hz có thể thay đổi thành phần chậm; 40 Hz loại thông tin tần số cao, không chỉ nhiễu. Zero-phase không có nghĩa giữ nguyên hình thái. Không dùng x[:, ::2] thay cho 500→200 Hz.

In [ ]:
import numpy as np
from scipy.signal import butter, sosfiltfilt, resample_poly

def processing_candidates(values, source_fs=500):
    if source_fs != 500:
        raise ValueError("Minh họa này yêu cầu raw 500 Hz")
    raw = np.asarray(values, dtype=np.float64)
    if raw.ndim != 2 or not np.isfinite(raw).all():
        raise ValueError("Cần tín hiệu hữu hạn [lead,time]")
    sos = butter(4, [0.5, 40], btype="bandpass", fs=source_fs, output="sos")
    return {
        "minimal_500": (raw.copy(), 500),
        "bandpass_0.5_40": (sosfiltfilt(sos, raw, axis=-1), 500),
        "resample_200": (resample_poly(raw, up=2, down=5, axis=-1), 200),
    }

In [ ]:
candidates = processing_candidates(raw_mV, source_fs=raw_record.fs)
display(pd.DataFrame([{"phuong_an": name, "shape": str(values.shape), "fs_Hz": fs, "thoi_luong_s": values.shape[1] / fs, "so_phan_tu_so_voi_raw": values.size / raw_mV.size} for name, (values, fs) in candidates.items()]))
lead_index = LEADS.index("II")
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
for name, (values, fs) in candidates.items():
    for axis in axes:
        axis.plot(np.arange(values.shape[1]) / fs, values[lead_index], label=name, linewidth=0.8)
axes[0].set(xlim=(0, 2), xlabel="Giây", ylabel="mV", title="Biên đầu và 2 giây đầu")
axes[1].set(xlim=(4, 6), xlabel="Giây", ylabel="mV", title="Khoảng giữa 4–6 giây")
axes[0].legend()
fig.suptitle(f"Cùng ECG train {int(example.ecg_id)}, lead II — chưa kết luận tốt hơn")
fig.tight_layout()
plt.show()
filtered = candidates["bandpass_0.5_40"][0]
difference = filtered - raw_mV
interior = slice(500, 4500)
display(pd.DataFrame({"lead": LEADS, "RMSE_thay_doi_toan_10s_mV": np.sqrt(np.mean(difference ** 2, axis=1)), "RMSE_thay_doi_1_9s_mV": np.sqrt(np.mean(difference[:, interior] ** 2, axis=1))}))
print("RMSE là độ thay đổi so raw, không phải sai số so tín hiệu sạch chuẩn.")

## 9. PSD giúp nhìn năng lượng, không tự quyết định băng thông
Welch chia cửa sổ chồng nhau và trung bình ước lượng phổ. Đặt fs đúng cho từng bản; cửa sổ 2 giây, overlap 50%, detrend constant, đơn vị mV²/Hz. Do detrend, không dùng hình này đo chính xác DC/đường nền rất chậm. Bản 200 Hz chỉ có phổ tới 100 Hz.

Vùng ít năng lượng vẫn có thể chứa đặc trưng hữu ích. Không chọn 40 Hz chỉ vì hình mượt hoặc năng lượng trên 40 Hz thấp.

In [ ]:
from scipy.signal import welch

fig, axis = plt.subplots(figsize=(10, 4))
for name, (values, fs) in candidates.items():
    nperseg = min(int(2 * fs), values.shape[1])
    frequencies, density = welch(values[lead_index], fs=fs, window="hann", nperseg=nperseg, noverlap=nperseg // 2, detrend="constant", scaling="density")
    axis.semilogy(frequencies, np.maximum(density, np.finfo(float).tiny), label=name)
axis.axvline(40, color="gray", linestyle="--", linewidth=0.8, label="40 Hz")
axis.set(xlim=(0, 120), xlabel="Tần số (Hz)", ylabel="PSD (mV²/Hz)", title=f"Welch — ECG train {int(example.ecg_id)}, lead II")
axis.legend()
axis.grid(alpha=0.2)
plt.show()

## Quyết định hiện tại và thí nghiệm sau
Giữ 500 Hz/minimal làm baseline có thể truy vết. Hình/thống kê mô tả chưa chứng minh bộ lọc nào cải thiện model. Tạo release ứng viên riêng khi chốt thí nghiệm; các phép tính trong RAM trên không ghi đè .npy.

| Câu hỏi | Giữ cố định | Đánh giá sau này |
|---|---|---|
| 0,5–40 Hz có giúp? | Cohort, patient split, model, seed, ngân sách train | Macro và từng nhãn AUROC/AUPRC trên validation; lỗi nhóm có/không chú thích nhiễu |
| 200 Hz có đủ? | Như trên; chuẩn hóa riêng từ train của mỗi release | Hiệu năng, thời gian, bộ nhớ thực đo; thẩm định hình thái cần chuyên môn |
| Loại nhiễu có gây thiên lệch? | Định nghĩa nhóm trước khi xem test | Số mẫu/nhãn mất đi, hiệu năng subgroup và bất định |

Test giữ lại tới khi chốt lựa chọn. Cần chuyên môn và dữ liệu ngoài để kết luận lâm sàng; QC kỹ thuật không thay thế chúng.

Nguồn đối chiếu 01/11/2026: [WFDB](https://wfdb.readthedocs.io/en/latest/io.html), [SOS hai chiều](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.sosfiltfilt.html), [Welch](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.welch.html), [resample_poly](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.resample_poly.html). Tiếp theo: **03_chuan_hoa_tu_train.ipynb**.